# Week 88, Day 2: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('walks.py', 'w', encoding='utf-8') as f:
    f.write('WALKS = [\n    {"day": "2026-01-04", "km": 6.0, "climb_m": 270, "minutes": 105, "route": "Ridge walk"},\n    {"day": "2026-01-18", "km": 7.5, "climb_m": 40, "minutes": 98, "route": None},\n    {"day": "2026-02-01", "km": 11.0, "climb_m": 110, "minutes": 150, "route": "Lake loop"},\n    {"day": "2026-02-08", "km": 6.0, "climb_m": 270, "minutes": 101, "route": "Ridge walk"},\n    {"day": "2026-02-15", "km": 4.0, "climb_m": 0, "minutes": 50, "route": None},\n    {"day": "2026-03-29", "km": 15.5, "climb_m": 135, "minutes": 215, "route": None},\n    {"day": "2026-04-05", "km": 5.5, "climb_m": 110, "minutes": 80, "route": None},\n    {"day": "2026-04-12", "km": 13.0, "climb_m": 270, "minutes": 195, "route": None},\n    {"day": "2026-04-19", "km": 6.0, "climb_m": 270, "minutes": 97, "route": "Ridge walk"},\n]\n')

**Predict before you run:** what will this print?



In [ ]:
from datetime import date

texts = ["2026-11-9", "2026-11-10", "2026-02-1"]
print(sorted(texts))
print(max(texts) > "2026-11-10")
print(date(2026, 11, 9) < date(2026, 11, 10), date.fromisoformat("2026-11-10").isocalendar()[:2])

In [ ]:
import sqlite3
from walks import WALKS

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE walk (day TEXT NOT NULL, km REAL NOT NULL CHECK (km > 0), climb_m INTEGER NOT NULL, minutes INTEGER NOT NULL)")
conn.executemany("INSERT INTO walk VALUES (:day, :km, :climb_m, :minutes)", WALKS)
rows = conn.execute('''SELECT strftime('%m', day) AS month, COUNT(*), SUM(km) FROM walk
                       WHERE strftime('%Y', day) = ? GROUP BY month ORDER BY month''', ("2026",)).fetchall()
print(rows)
by_month = {int(m): (n, km) for m, n, km in rows}
print([by_month.get(m, (0, 0.0))[1] for m in range(1, 13)])

In [ ]:
import statistics
from waymark_domain import naismith_minutes
from walks import WALKS

ratios = [w["minutes"] / naismith_minutes(w["km"], w["climb_m"]) for w in WALKS]
print([round(r, 2) for r in ratios])
print("mean", round(statistics.mean(ratios), 3), "median", round(statistics.median(ratios), 3))
print("Ridge walk for this walker:", round(naismith_minutes(6, 270) * statistics.median(ratios)), "minutes, not", naismith_minutes(6, 270))

> **Key idea.** Validate records by collecting every problem; parse text into real types (`date.fromisoformat`) before checking; inject "today" rather than calling `date.today()` inside logic; normalise what you store. Summarise in SQL with `strftime` and `GROUP BY`, and fill empty periods in Python. Personalise estimates with the median of a person's own `actual / estimate` ratios, falling back to the general rule when there's too little history. Identify weeks by their Monday (or ISO year and week), never by week number alone, so streaks survive the new year.

### Exercise: Validating a walk

Write `check_walk(data, today)`: `data` is a dict with `day` (text), `km`, `climb_m`, `minutes` and optionally `route`. Return `(walk, problems)`: `problems` lists, in this order, `"day must be a date like 2026-11-14"` (if it doesn't parse with `date.fromisoformat`), `"day is in the future"`, `"km must be a positive number"` (a number above 0, not a boolean), `"climb_m must be a whole number of metres, 0 or more"`, `"minutes must be between 1 and 1440"` (a whole number), and `"route must be text"` (if present and not `None` or a string). If there are problems, `walk` is `None`; otherwise it's the normalised dict: `day` as `YYYY-MM-DD` text, `km` rounded to 2 places, `climb_m`, `minutes` and `route` (or `None`).

In [ ]:
from datetime import date

def check_walk(data, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date\n\nTODAY = date(2026, 11, 14)\n\ndef test_valid():\n    \"parsed and normalised\"\n    walk, problems = check_walk({\"day\": \"2026-11-14\", \"km\": 6.004, \"climb_m\": 270, \"minutes\": 101}, TODAY)\n    assert problems == [] and walk == {\"day\": \"2026-11-14\", \"km\": 6.0, \"climb_m\": 270, \"minutes\": 101, \"route\": None}\n    assert check_walk({\"day\": \"2026-01-04\", \"km\": 3, \"climb_m\": 0, \"minutes\": 40, \"route\": \"Lake loop\"}, TODAY)[0][\"route\"] == \"Lake loop\"\n\ndef test_problems():\n    \"every problem at once\"\n    walk, problems = check_walk({\"day\": \"2026-11-15\", \"km\": 0, \"climb_m\": -5, \"minutes\": 2000, \"route\": 7}, TODAY)\n    assert walk is None and problems == [\"day is in the future\", \"km must be a positive number\",\n                                         \"climb_m must be a whole number of metres, 0 or more\",\n                                         \"minutes must be between 1 and 1440\", \"route must be text\"]\n    assert check_walk({\"day\": \"14/11/2026\", \"km\": True, \"climb_m\": 1.5, \"minutes\": 30}, TODAY)[1] == [\n        \"day must be a date like 2026-11-14\", \"km must be a positive number\", \"climb_m must be a whole number of metres, 0 or more\"]\n    assert check_walk({\"km\": 1, \"climb_m\": 0, \"minutes\": 10}, TODAY)[1] == [\"day must be a date like 2026-11-14\"]")

### Exercise: A year by month

Write `WalkLog(conn)`, creating `walk (id INTEGER PRIMARY KEY, day TEXT NOT NULL, km REAL NOT NULL CHECK (km > 0), climb_m INTEGER NOT NULL CHECK (climb_m >= 0), minutes INTEGER NOT NULL CHECK (minutes > 0), route TEXT)`. `add(walk)` stores a walk dict and returns its id. `months(year)` returns 12 tuples `(month, walks, km, climb_m, minutes)` for months 1 to 12, with zeros for months with no walks and km rounded to 1 place, using one SQL query with `GROUP BY`. `total_km(year)` returns the year's distance rounded to 1 place.

In [ ]:
import sqlite3

class WalkLog:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\nfrom walks import WALKS\n\ndef test_log():\n    \"every month, gaps filled\"\n    log = WalkLog(sqlite3.connect(\":memory:\"))\n    ids = [log.add(w) for w in WALKS]\n    assert ids == list(range(1, 10))\n    months = log.months(2026)\n    assert len(months) == 12 and months[0] == (1, 2, 13.5, 310, 203) and months[2] == (3, 1, 15.5, 135, 215)\n    assert months[3] == (4, 3, 24.5, 650, 372) and months[4] == (5, 0, 0.0, 0, 0)\n    assert log.total_km(2026) == 74.5 and log.total_km(2025) == 0.0\n    assert \"GROUP BY\" in source.upper()")

### Exercise: A personal pace

Using `naismith_minutes` from the page's `waymark_domain`, write `pace_factor(walks, minimum=3)`: the median of `minutes / naismith_minutes(km, climb_m)` over the walks, rounded to 2 places, or `1.0` if there are fewer than `minimum` walks. Then `personal_minutes(km, climb_m, walks)`: the rule's estimate times the pace factor, rounded to a whole number.

In [ ]:
import statistics
from waymark_domain import naismith_minutes

def pace_factor(walks, minimum=3):
    ___

def personal_minutes(km, climb_m, walks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from walks import WALKS\n\ndef test_pace():\n    \"the median of your own walks\"\n    assert pace_factor(WALKS) == 1.04\n    assert pace_factor(WALKS[:2]) == 1.0\n    assert personal_minutes(6, 270, WALKS) == 103 and personal_minutes(6, 270, []) == 99\n    slow = WALKS + [{\"day\": \"2026-05-01\", \"km\": 5.0, \"climb_m\": 0, \"minutes\": 600, \"route\": None}]\n    assert pace_factor(slow) == pace_factor(WALKS) or abs(pace_factor(slow) - pace_factor(WALKS)) < 0.05, \"one long lunch barely matters\"")

### Exercise: Weekly streaks

Write `week_start(day)`: the Monday of the week containing a `date`. Then `longest_streak(days)`: the most consecutive weeks (Monday to Sunday) that each contain at least one of the given dates; and `current_streak(days, today)`: the length of the streak of consecutive weeks that ends with **this** week or **last** week (so a streak isn't broken until a whole week passes without a walk), or 0.

In [ ]:
from datetime import date, timedelta

def week_start(day):
    ___

def longest_streak(days):
    ___

def current_streak(days, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date\n\ndef d(text):\n    return date.fromisoformat(text)\n\ndef test_streaks():\n    \"weeks by their Monday, across the new year\"\n    assert week_start(d(\"2026-11-15\")) == d(\"2026-11-09\") and week_start(d(\"2026-11-09\")) == d(\"2026-11-09\")\n    days = [d(x) for x in [\"2025-12-22\", \"2025-12-30\", \"2026-01-02\", \"2026-01-07\", \"2026-01-25\", \"2026-02-01\"]]\n    assert longest_streak(days) == 3, \"the weeks of Dec 22, Dec 29 and Jan 5\"\n    assert longest_streak([]) == 0 and longest_streak([d(\"2026-03-03\")]) == 1\n    assert current_streak(days, d(\"2026-02-03\")) == 2\n    assert current_streak(days, d(\"2026-02-08\")) == 2, \"last week still counts\"\n    assert current_streak(days, d(\"2026-02-16\")) == 0")

### Bug hunt: November the ninth is later than the tenth

The days were typed without leading zeros and are compared as text, so the latest walk comes out wrong: it prints `latest: 2026-11-9`. Parse them as dates (`date(*map(int, text.split("-")))` copes with missing zeros) and print the latest as ISO text, so it prints `latest: 2026-11-10`.

In [ ]:
from datetime import date

typed = ["2026-11-9", "2026-11-10", "2026-2-1"]
print("latest:", max(typed))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"parse before comparing\"\n    expect_output(\"latest: 2026-11-10\")")

### Bug hunt: A streak broken by New Year

Comparing ISO week **numbers** breaks at the end of the year (week 1 follows week 52 or 53), so this streak across New Year comes out as `streak: 1`. Compare the weeks' Mondays instead, so it prints `streak: 3`.

In [ ]:
from datetime import date

days = [date(2025, 12, 22), date(2025, 12, 31), date(2026, 1, 6)]
weeks = sorted({d.isocalendar()[1] for d in days})
streak = best = 1
for a, b in zip(weeks, weeks[1:]):
    streak = streak + 1 if b - a == 1 else 1
    best = max(best, streak)
print("streak:", best)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Mondays, not week numbers\"\n    expect_output(\"streak: 3\")")

### Puzzle: Logging a walk

Arrange the service that logs a walk: check it with an injected today, refuse with every problem, otherwise store it and return the stored walk. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from datetime import date
p = ___   # sorted(["2026-11-9", "2026-11-10"])[0]
q = ___   # date(2026, 1, 1).isocalendar()[1]  (ISO week number of New Year's Day 2026)
r = ___   # the median of [1.0, 1.1, 1.2, 5.0]
s = ___   # the Monday (as ISO text) of the week containing 2026-11-15, a Sunday
t = ___   # True or False: logic should call date.today() itself

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"2026-11-10\" and q == 1 and r == 1.15, \"p, q and r\"\n    assert s == \"2026-11-09\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the year in walks

Write `year_report(walks, year, today)` for a list of walk dicts like those in `walks.py`, using the given helpers. Only walks in `year` count. Return text exactly like the example in the checks: a title, the number of walks, total distance (1 place) and climb, total time as hours and minutes, the longest walk (its distance, day and route, or `no route` when it has none), the busiest month by distance (its full English name, from `calendar.month_name`, ties to the earlier month), the pace factor as a percentage difference from Naismith's rule (`"4% slower than Naismith's rule"`, `"4% faster ..."` or `"exactly as Naismith's rule"`), and the longest and current weekly streaks. With no walks that year, return the title line and `No walks logged.`.

In [ ]:
import calendar
import statistics
from datetime import date, timedelta
from waymark_domain import naismith_minutes

def pace_factor(walks, minimum=3):
    if len(walks) < minimum:
        return 1.0
    return round(statistics.median(w["minutes"] / naismith_minutes(w["km"], w["climb_m"]) for w in walks), 2)

def week_start(day):
    return day - timedelta(days=day.weekday())

def streaks(days, today):
    weeks = sorted({week_start(d) for d in days})
    best = run = 0
    for i, w in enumerate(weeks):
        run = run + 1 if i and w - weeks[i - 1] == timedelta(days=7) else 1
        best = max(best, run)
    current = run if weeks and week_start(today) - weeks[-1] <= timedelta(days=7) else 0
    return best, current

def year_report(walks, year, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date\nfrom walks import WALKS\n\ndef test_report():\n    \"a report worth reading\"\n    assert year_report(WALKS, 2026, date(2026, 4, 25)) == \"\"\"Waymark: your 2026 in walks\nWalks: 9\nDistance: 74.5 km, climb 1475 m\nTime on foot: 18 h 11 min\nLongest walk: 15.5 km on 2026-03-29 (no route)\nBusiest month: April (24.5 km)\nPace: 4% slower than Naismith's rule\nWeekly streaks: longest 4, current 4\n\"\"\"\n    assert year_report(WALKS, 2025, date(2026, 4, 25)) == \"Waymark: your 2025 in walks\\nNo walks logged.\\n\"\n\ndef test_variations():\n    \"faster walkers and named routes\"\n    quick = [{\"day\": \"2026-06-0%d\" % d, \"km\": 10.0, \"climb_m\": 0, \"minutes\": 115, \"route\": \"Canal\"} for d in (1, 2, 3)]\n    text = year_report(quick, 2026, date(2026, 7, 1))\n    assert \"Pace: 4% faster than Naismith's rule\" in text and \"Longest walk: 10.0 km on 2026-06-01 (Canal)\" in text\n    assert \"Weekly streaks: longest 1, current 0\" in text\n    exact = [{\"day\": \"2026-06-0%d\" % d, \"km\": 10.0, \"climb_m\": 0, \"minutes\": 120, \"route\": None} for d in (1, 2, 3)]\n    assert \"Pace: exactly as Naismith's rule\" in year_report(exact, 2026, date(2026, 6, 4))")

## Warm-up

Text sorts character by character, so `"2026-11-9"` comes after `"2026-11-10"`. Dates that are always written `YYYY-MM-DD` with zeros happen to sort correctly as text, which is why databases store them that way; but anything typed by a person must be **parsed** into a `date` first, and rejected if it can't be.

## Validating a walk

A walk record has a day, a distance, a climb, the minutes it took, and optionally the saved route it followed. Validation should:

- **collect every problem**, not stop at the first, so the walker can fix them all at once: return a list of messages, empty if all is well;
- **parse, then check**: turn the day into a `date` (`date.fromisoformat`), and the numbers into numbers, before checking ranges;
- **check against an injected `today`**, never `date.today()` inside the function, so tests can say what today is and the "no walks in the future" rule can be tested on any day;
- **normalise** what it accepts (a `date` back to a `YYYY-MM-DD` string, distances rounded), so everything stored looks the same.

## Storing and summarising

Walks go in a `walk` table, with `CHECK` constraints repeating the most important rules (Day 3's second line of defence). Summaries are SQL's job: `strftime('%Y-%m', day)` gives each walk's month, and `GROUP BY` totals them. SQL only returns months that **have** walks, so a year's summary fills the missing months with zeros in Python: a chart with a gap where February should be is misleading.

## A personal pace

Naismith's rule is for an average walker. Waymark knows better: every logged walk has an estimate (from its distance and climb) and an actual time. The ratio `actual / estimate` says how this walker compares: 1.1 means 10% slower than the rule. Use the **median** of the ratios, not the mean, so one walk with a long lunch stop doesn't change every future estimate. With too few walks (fewer than three, say), there isn't enough evidence: fall back to the rule.

## Streaks and weeks

"Walked every week for six weeks" is motivating, and easy to get wrong. A week is identified by its **ISO year and week number**, `date.isocalendar()`, because the last days of December can belong to week 1 of the next year and the first days of January to week 52 or 53 of the last. The safest way to test "consecutive weeks" avoids week numbers altogether: two walks are in consecutive weeks if the **Mondays** of their weeks are exactly 7 days apart (`day - timedelta(days=day.weekday())` is that Monday).

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Dates and times in your project**: find every place you compare, store or display them. Are they parsed? Is "now" injected? Fix one.
2. **Build today's story** inside out, with its scenarios as tests, including at least one that depends on what "today" is.
3. **A summary**: if your project has records over time, add one honest summary (with empty periods filled).
4. **Merge** and update the burndown.

## Before you go

Look at your burndown after two build days: on track, ahead or behind? If behind, decide today what to drop from the sprint, not on Friday. Tomorrow is S6 and S8: importing a map from a file, and finding everything wrong with it.